In [1]:
"""
Calcul des scores des modèles sur short_answers_final.xlsx (réponses courtes / short answers)
-----------------------------------------------------------------------------------------------
V2 — Scoring par JUGE LLM (Gemma 4) au lieu de l'inclusion/similarité floue : trop de bonnes
réponses étaient mal notées par un matching textuel (formulations différentes, reformulations,
langues différentes, etc.). On délègue maintenant le jugement sémantique à Gemma 4, qui reçoit
la question, la/les réponse(s) de référence acceptée(s), et la réponse du modèle à évaluer, et
qui doit répondre en verdict binaire : CORRECT ou INCORRECT.

Détecte AUTOMATIQUEMENT toutes les colonnes "response_<modele>" présentes dans la feuille (sauf
d'éventuelles colonnes techniques _raw / _attempts) -> aucune modification nécessaire si tu
ajoutes de nouveaux modèles, il suffit de relancer la cellule.

Fonctionnement du juge :
  - Un item sans réponse du modèle (NaN / vide) n'est PAS envoyé au juge -> statut "manquant".
  - Chaque appel envoie : la question, les variantes acceptées (answer + primary + second,
    séparées par '|'), et la réponse à évaluer. Gemma doit répondre STRICTEMENT au format
    "VERDICT: CORRECT" ou "VERDICT: INCORRECT" (+ justification optionnelle en une phrase).
  - Le texte renvoyé est parsé par regex ; si aucun verdict n'est trouvé -> statut "non_parse".
    Si l'appel réseau échoue après plusieurs tentatives -> statut "erreur_api".
  - Un CACHE DISQUE (JSON) mémorise chaque verdict par (item_id, modèle, réponse exacte) pour
    ne jamais repayer/rappeler deux fois le même jugement, y compris si tu relances la cellule
    après une interruption. Le cache est sauvegardé régulièrement pendant l'exécution.
  - Les appels sont parallélisés (ThreadPoolExecutor) car c'est un juge distant (I/O bound).

Produit un classeur Excel avec plusieurs feuilles :
  - Notes                  : explication des feuilles et des métriques
  - Dataset                : taille et répartition du jeu de données évalué
  - Overview                : classement des modèles (exactitude, couverture, taux d'erreurs)
  - Acc_Axe / N_Axe                    : exactitude / effectifs par Axe
  - Acc_Difficulty / N_Difficulty      : exactitude / effectifs par Difficulty
  - Acc_intent_group / N_intent_group  : exactitude / effectifs par intent_group
  - Acc_intent_category / N_intent_category : exactitude / effectifs par intent_category
  - Details                 : chaque item + réponse de chaque modèle + statut + justification
                               du juge (pratique pour auditer les verdicts à la main).
"""

import hashlib
import json
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import urllib3

from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ============================== CONFIG DONNÉES ==============================

INPUT_PATH = "short_answers_final.xlsx"
INPUT_SHEET = "Feuil1"   # si absente, la seule feuille du classeur est utilisée automatiquement
OUTPUT_PATH = "short_answers_scores_llm_judge.xlsx"

GROUND_TRUTH_COL = "answer"
PRIMARY_VARIANTS_COL = "primary_accepted_variants"
SECOND_VARIANTS_COL = "second_accepted_variants"
VARIANT_SEP = "|"
QUESTION_COL = "input"
ITEM_ID_COL = "item_id"

STATUS_COL = "status"                   # ignoré si la colonne n'existe pas
VALID_STATUSES = {"accepted"}           # mets None pour désactiver le filtre par statut

MODEL_COL_PREFIX = "response_"
NON_MODEL_SUFFIXES = ("_raw", "_attempts")

GROUP_COLS = ["Axe", "Difficulty", "intent_group", "intent_category"]
DIFFICULTY_ORDER = ["easy", "medium", "hard"]
EXTRA_DETAIL_COLS = ["all_supporting_sources"]

LIMIT_ROWS = None   # ex: 20 pour tester rapidement le pipeline sur un petit échantillon
                     # avant de lancer le run complet (économise des appels API pendant les tests)

# ============================== CONFIG JUGE (GEMMA 4) ==============================

GEMMA_URL = "https://vllm-gemma4-31b-v2-mtrna-ns1.apps.olympus.atlasxai.ma/v1/chat/completions"
GEMMA_MODEL = "gemma-4-31B-v2"
# ⚠️ à terme, préfère charger la clé depuis une variable d'environnement plutôt que de la
# laisser en dur ici (ex: os.environ["GEMMA_API_KEY"]).
GEMMA_API_KEY = "91eb20660c62920a5ebee5ab782659923c5077b858b2dd1545f978d02eb63084"

REQUEST_TIMEOUT = 60        # secondes
MAX_RETRIES = 4
RETRY_BACKOFF_BASE = 2.0    # secondes, multiplié par le numéro de tentative
MAX_WORKERS = 8             # nombre d'appels en parallèle vers le juge

CACHE_PATH = "gemma_judge_cache.json"
CACHE_SAVE_EVERY = 25       # sauvegarde du cache tous les N appels réseau
PROMPT_VERSION = "v1"       # change cette valeur si tu modifies le prompt, pour invalider le cache

HEADERS = {
    "Authorization": f"Bearer {GEMMA_API_KEY}",
    "Content-Type": "application/json",
}

SYSTEM_PROMPT = """Tu es un correcteur expert et rigoureux, chargé d'évaluer si la réponse d'un \
modèle d'IA à une question de culture générale marocaine (histoire, géographie, société, etc., \
souvent posée en darija) est correcte.

Tu compares le SENS de la réponse à évaluer avec une réponse de référence (qui peut être donnée \
sous plusieurs formulations acceptées). Ignore les différences de langue (arabe/darija/français/\
anglais), d'orthographe, de formulation, ou la présence d'informations supplémentaires non \
contradictoires.

Réponds CORRECT si la réponse évaluée correspond sémantiquement à la réponse de référence, ou \
n'en diffère que de façon mineure et sans incidence sur le sens (ex: variante orthographique, \
synonyme, date quasi identique).
Réponds INCORRECT si la réponse évaluée est fausse, hors sujet, un refus de répondre, une \
hallucination, ou contredit la réponse de référence.

Réponds STRICTEMENT selon ce format, rien d'autre avant :
VERDICT: CORRECT
ou
VERDICT: INCORRECT
Tu peux ajouter une très courte justification (une phrase) sur la ligne suivante."""

USER_TEMPLATE = """Question : {question}

Réponse(s) de référence acceptée(s) : {variants_str}

Réponse à évaluer : {response}"""

_VERDICT_RE = re.compile(r"VERDICT\s*:\s*(CORRECT|INCORRECT)", re.IGNORECASE)


# ============================== APPEL AU JUGE ==============================

def build_judge_prompt(question: str, variants_str: str, response: str) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(
            question=question, variants_str=variants_str, response=response
        )},
    ]


def call_gemma_judge(session: requests.Session, messages: list) -> str:
    payload = {
        "model": GEMMA_MODEL,
        "messages": messages,
        "temperature": 0,
        "max_tokens": 150,
    }
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.post(
                GEMMA_URL, headers=HEADERS, json=payload,
                timeout=REQUEST_TIMEOUT, verify=False,
            )
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_BASE * attempt)
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives : {last_err}")


def parse_judge_output(text: str):
    """Retourne (status, reason). status ∈ {'correct', 'incorrect', 'non_parse'}."""
    if not text or not text.strip():
        return "non_parse", ""
    match = _VERDICT_RE.search(text)
    if match:
        verdict = match.group(1).upper()
        status = "correct" if verdict == "CORRECT" else "incorrect"
        reason = text[match.end():].strip().splitlines()
        reason = reason[0].strip(" -:\t") if reason else ""
        return status, reason[:300]

    upper = text.upper()
    if "INCORRECT" in upper:
        return "incorrect", text.strip()[:300]
    if "CORRECT" in upper:
        return "correct", text.strip()[:300]
    return "non_parse", text.strip()[:300]


# ============================== CACHE DISQUE ==============================

def load_cache() -> dict:
    p = Path(CACHE_PATH)
    if not p.exists():
        return {}
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Cache illisible ({e}), on repart d'un cache vide.")
        return {}


def save_cache(cache: dict):
    tmp = Path(CACHE_PATH).with_suffix(".tmp.json")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    tmp.replace(CACHE_PATH)


def build_cache_key(item_id, model_col: str, response_text: str) -> str:
    raw = f"{PROMPT_VERSION}|{item_id}|{model_col}|{response_text}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


# ============================== HELPERS DONNÉES ==============================

def model_label(col: str) -> str:
    return col[len(MODEL_COL_PREFIX):]


def detect_model_columns(df: pd.DataFrame) -> list:
    cols = [
        c for c in df.columns
        if c.startswith(MODEL_COL_PREFIX) and not c.endswith(NON_MODEL_SUFFIXES)
    ]
    return sorted(cols)


def order_group_values(group_col: str, series: pd.Series) -> list:
    values = series.dropna().unique().tolist()
    if group_col.lower() == "difficulty":
        lower_map = {str(v).strip().lower(): v for v in values}
        ordered = [lower_map[d] for d in DIFFICULTY_ORDER if d in lower_map]
        remaining = sorted([v for v in values if v not in ordered], key=str)
        return ordered + remaining
    counts = series.value_counts()
    return sorted(values, key=lambda v: (-counts.get(v, 0), str(v)))


def get_accepted_variants_text(row: pd.Series) -> list:
    """Liste dédupliquée (par forme stricte en minuscules) des formulations acceptées :
    answer + primary_accepted_variants + second_accepted_variants."""
    vals = []
    gt = row.get(GROUND_TRUTH_COL)
    if pd.notna(gt) and str(gt).strip():
        vals.append(str(gt).strip())
    for col in (PRIMARY_VARIANTS_COL, SECOND_VARIANTS_COL):
        raw = row.get(col)
        if pd.notna(raw):
            vals.extend([p.strip() for p in str(raw).split(VARIANT_SEP) if p.strip()])

    seen, out = set(), []
    for v in vals:
        key = v.lower()
        if key not in seen:
            seen.add(key)
            out.append(v)
    return out


def load_eligible_data():
    if not Path(INPUT_PATH).exists():
        raise FileNotFoundError(f"Fichier introuvable : {INPUT_PATH}")

    xls = pd.ExcelFile(INPUT_PATH)
    sheet = INPUT_SHEET if INPUT_SHEET in xls.sheet_names else None
    if sheet is None:
        if len(xls.sheet_names) == 1:
            sheet = xls.sheet_names[0]
            print(f"Feuille '{INPUT_SHEET}' introuvable, utilisation de l'unique feuille : '{sheet}'")
        else:
            raise ValueError(
                f"Feuille '{INPUT_SHEET}' introuvable. Feuilles disponibles : {xls.sheet_names}. "
                f"Mets à jour INPUT_SHEET."
            )
    df = pd.read_excel(INPUT_PATH, sheet_name=sheet)

    if GROUND_TRUTH_COL not in df.columns:
        raise ValueError(
            f"Colonne de vérité terrain '{GROUND_TRUTH_COL}' introuvable dans la feuille "
            f"'{sheet}'. Colonnes disponibles : {list(df.columns)}"
        )

    model_cols = detect_model_columns(df)
    if not model_cols:
        raise ValueError(
            f"Aucune colonne de réponse de modèle détectée (préfixe '{MODEL_COL_PREFIX}', "
            f"hors suffixes {NON_MODEL_SUFFIXES}). Vérifie le nom des colonnes."
        )

    n_before = len(df)
    if STATUS_COL in df.columns and VALID_STATUSES is not None:
        df = df[df[STATUS_COL].isin(VALID_STATUSES)]
        print(f"Filtre statut ({VALID_STATUSES}) : {n_before} -> {len(df)} lignes.")
    else:
        print(f"Pas de filtre par statut appliqué ({len(df)} lignes).")

    n_before_gt = len(df)
    df = df[df[GROUND_TRUTH_COL].notna()]
    if len(df) < n_before_gt:
        print(f"{n_before_gt - len(df)} ligne(s) sans '{GROUND_TRUTH_COL}' exclue(s) (pas encore validées).")

    df = df.reset_index(drop=True)

    if LIMIT_ROWS is not None:
        df = df.head(LIMIT_ROWS).reset_index(drop=True)
        print(f"LIMIT_ROWS actif -> échantillon limité à {len(df)} lignes pour ce run.")

    print(f"-> {len(df)} items éligibles pour le scoring.")
    print(f"-> {len(model_cols)} modèle(s) détecté(s) : {[model_label(c) for c in model_cols]}")

    return df, model_cols


# ============================== SCORING PAR LE JUGE ==============================

def score_all_models_llm(df: pd.DataFrame, model_cols: list) -> dict:
    cache = load_cache()
    n = len(df)

    scored = {c: np.zeros(n, dtype=bool) for c in model_cols}
    status = {c: np.array(["manquant"] * n, dtype=object) for c in model_cols}
    reason = {c: np.array([""] * n, dtype=object) for c in model_cols}

    tasks = []  # (row_pos, col, messages, cache_key)
    for row_pos, (_, row) in enumerate(df.iterrows()):
        variants = get_accepted_variants_text(row)
        variants_str = " | ".join(variants) if variants else "(aucune)"
        item_id = row.get(ITEM_ID_COL, row_pos)
        question = str(row.get(QUESTION_COL, "")).strip()

        for col in model_cols:
            resp = row[col]
            if pd.isna(resp) or not str(resp).strip():
                continue  # reste "manquant", pas d'appel au juge
            resp_str = str(resp).strip()
            scored[col][row_pos] = True
            cache_key = build_cache_key(item_id, col, resp_str)

            if cache_key in cache:
                r = cache[cache_key]
                status[col][row_pos] = r["status"]
                reason[col][row_pos] = r.get("reason", "")
            else:
                messages = build_judge_prompt(question, variants_str, resp_str)
                tasks.append((row_pos, col, messages, cache_key))

    n_scored_total = sum(int(scored[c].sum()) for c in model_cols)
    print(f"{n_scored_total} réponse(s) non vide(s) au total ; {len(tasks)} appel(s) réseau "
          f"nécessaires (le reste vient du cache '{CACHE_PATH}').")

    if tasks:
        session = requests.Session()
        lock = threading.Lock()
        done_count = 0

        def worker(task):
            row_pos, col, messages, cache_key = task
            try:
                raw_text = call_gemma_judge(session, messages)
                st, rs = parse_judge_output(raw_text)
            except Exception as e:
                st, rs, raw_text = "erreur_api", str(e)[:300], ""
            return row_pos, col, cache_key, st, rs, raw_text

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(worker, t) for t in tasks]
            for fut in as_completed(futures):
                row_pos, col, cache_key, st, rs, raw_text = fut.result()
                status[col][row_pos] = st
                reason[col][row_pos] = rs
                cache[cache_key] = {"status": st, "reason": rs, "raw": raw_text}
                with lock:
                    done_count += 1
                    if done_count % CACHE_SAVE_EVERY == 0:
                        save_cache(cache)
                        print(f"   ... {done_count}/{len(tasks)} appels traités (cache sauvegardé)")

        save_cache(cache)
    print("Jugement terminé.")

    results = {}
    for col in model_cols:
        correct = np.array([s == "correct" for s in status[col]])
        correct_or_na = np.where(scored[col], correct, pd.NA)
        results[col] = {
            "scored": pd.Series(scored[col], index=df.index),
            "correct": pd.Series(correct_or_na, index=df.index),
            "status": pd.Series(status[col], index=df.index),
            "reason": pd.Series(reason[col], index=df.index),
        }
    return results


def compute_overview(df: pd.DataFrame, model_cols: list, results: dict) -> pd.DataFrame:
    n_total = len(df)
    rows = []

    for col in model_cols:
        name = model_label(col)
        scored = results[col]["scored"]
        status = results[col]["status"]

        n_scored = int(scored.sum())
        n_missing = n_total - n_scored
        n_correct = int((status == "correct").sum())
        n_incorrect = int((status == "incorrect").sum())
        n_non_parse = int((status == "non_parse").sum())
        n_api_error = int((status == "erreur_api").sum())

        accuracy = n_correct / n_scored if n_scored else np.nan

        rows.append({
            "modele": name,
            "n_total": n_total,
            "n_scored": n_scored,
            "n_correct": n_correct,
            "n_incorrect": n_incorrect,
            "couverture": n_scored / n_total if n_total else np.nan,
            "exactitude": accuracy,
            "taux_manquant": n_missing / n_total if n_total else np.nan,
            "taux_non_parse_juge": n_non_parse / n_total if n_total else np.nan,
            "taux_erreur_api": n_api_error / n_total if n_total else np.nan,
        })

    out = pd.DataFrame(rows).sort_values("exactitude", ascending=False, na_position="last").reset_index(drop=True)
    ranks = np.where(out["exactitude"].notna(), np.arange(1, len(out) + 1), np.nan)
    out.insert(0, "rang", ranks)
    return out


def compute_breakdown(df: pd.DataFrame, model_cols: list, group_col: str,
                        model_order: list, results: dict):
    acc_data, n_data = {}, {}

    for col in model_cols:
        name = model_label(col)
        scored = results[col]["scored"]
        status = results[col]["status"]
        correct = status == "correct"

        tmp = pd.DataFrame({"group": df[group_col], "scored": scored, "correct": correct})
        grouped = tmp.groupby("group", dropna=False)
        n_scored_per_group = grouped["scored"].sum()
        n_correct_per_group = grouped["correct"].sum()
        acc_data[name] = (n_correct_per_group / n_scored_per_group).replace([np.inf, -np.inf], np.nan)
        n_data[name] = n_scored_per_group

    acc_df = pd.DataFrame(acc_data).T.reindex(model_order)
    n_df = pd.DataFrame(n_data).T.reindex(model_order)

    col_order = order_group_values(group_col, df[group_col])
    acc_df = acc_df.reindex(columns=col_order)
    n_df = n_df.reindex(columns=col_order).fillna(0).astype(int)

    return acc_df.reset_index().rename(columns={"index": "modele"}), \
           n_df.reset_index().rename(columns={"index": "modele"})


def build_details(df: pd.DataFrame, model_cols: list, results: dict) -> pd.DataFrame:
    base_cols = [c for c in (
        [ITEM_ID_COL, "Axe", "Difficulty", "intent_group", "intent_category", QUESTION_COL,
         GROUND_TRUTH_COL, PRIMARY_VARIANTS_COL, SECOND_VARIANTS_COL, STATUS_COL]
        + EXTRA_DETAIL_COLS
    ) if c in df.columns]

    details = df[base_cols].copy()

    for col in model_cols:
        name = model_label(col)
        details[col] = df[col]
        details[f"{name}_statut"] = results[col]["status"]
        details[f"{name}_correct"] = results[col]["status"] == "correct"
        details[f"{name}_raison_juge"] = results[col]["reason"]

    return details


def build_dataset_summary(df: pd.DataFrame, group_cols_present: list) -> pd.DataFrame:
    total = len(df)
    rows = []
    for gc in group_cols_present:
        for val, n in df[gc].value_counts(dropna=False).items():
            rows.append({"categorie": gc, "valeur": str(val), "n": int(n),
                         "part": n / total if total else np.nan})
    out = pd.DataFrame(rows)
    out.insert(0, "n_total_dataset", total)
    return out


# ============================== MISE EN FORME EXCEL ==============================

HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", bold=True)


def style_header(ws, n_cols: int):
    for c in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=c)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")


def autofit_columns(ws, df: pd.DataFrame):
    for i, col in enumerate(df.columns, start=1):
        try:
            max_len = max([len(str(col))] + [len(str(v)) for v in df[col].astype(str).tolist()])
        except Exception:
            max_len = len(str(col))
        ws.column_dimensions[get_column_letter(i)].width = min(max_len + 3, 45)


def apply_percent_format(ws, df: pd.DataFrame, percent_cols: list):
    for col_name in percent_cols:
        if col_name not in df.columns:
            continue
        col_idx = df.columns.get_loc(col_name) + 1
        for row in range(2, len(df) + 2):
            ws.cell(row=row, column=col_idx).number_format = "0.0%"


def apply_color_scale(ws, df: pd.DataFrame, cols: list, invert: bool = False):
    if len(df) == 0:
        return
    colors = ("F8696B", "FFEB84", "63BE7B") if not invert else ("63BE7B", "FFEB84", "F8696B")
    for col_name in cols:
        if col_name not in df.columns:
            continue
        col_idx = df.columns.get_loc(col_name) + 1
        letter = get_column_letter(col_idx)
        cell_range = f"{letter}2:{letter}{len(df) + 1}"
        rule = ColorScaleRule(
            start_type="min", start_color=colors[0],
            mid_type="percentile", mid_value=50, mid_color=colors[1],
            end_type="max", end_color=colors[2],
        )
        ws.conditional_formatting.add(cell_range, rule)


def write_sheet(writer, df: pd.DataFrame, sheet_name: str,
                 percent_cols=None, color_scale_cols=None, invert_cols=None):
    df.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]
    n_rows, n_cols = df.shape
    style_header(ws, n_cols)
    autofit_columns(ws, df)
    if n_rows > 0:
        ws.auto_filter.ref = ws.dimensions
        ws.freeze_panes = "B2"
    if percent_cols:
        apply_percent_format(ws, df, percent_cols)
    if color_scale_cols:
        invert_cols = invert_cols or []
        apply_color_scale(ws, df, [c for c in color_scale_cols if c not in invert_cols], invert=False)
        apply_color_scale(ws, df, [c for c in color_scale_cols if c in invert_cols], invert=True)


def write_notes_sheet(writer, model_cols: list, group_cols_present: list):
    wb = writer.book
    ws = wb.create_sheet("Notes", 0)
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 100

    def line(row, a, b="", bold=False, size=11):
        ca = ws.cell(row=row, column=1, value=a)
        cb = ws.cell(row=row, column=2, value=b)
        if bold:
            ca.font = Font(bold=True, size=size)
            cb.font = Font(bold=True, size=size)
        ca.alignment = Alignment(wrap_text=True, vertical="top")
        cb.alignment = Alignment(wrap_text=True, vertical="top")

    r = 1
    line(r, "Score des modèles — Short Answers (juge LLM)", bold=True, size=14); r += 2
    line(r, "Modèles détectés", ", ".join(model_label(c) for c in model_cols)); r += 1
    line(r, "Dimensions de répartition", ", ".join(group_cols_present) if group_cols_present else "aucune"); r += 1
    line(r, "Juge utilisé", f"{GEMMA_MODEL} ({GEMMA_URL})"); r += 2

    line(r, "Méthode de scoring", bold=True); r += 1
    line(r, "", "Pour chaque réponse non vide d'un modèle, Gemma 4 reçoit la question, les "
               "formulations acceptées (answer + primary_accepted_variants + "
               "second_accepted_variants) et la réponse à évaluer, puis rend un verdict binaire "
               "CORRECT / INCORRECT en jugeant le SENS de la réponse (différences de langue, "
               "d'orthographe ou de formulation ignorées)."); r += 1
    line(r, "", "Les verdicts sont mis en cache localement (fichier JSON) par (item, modèle, "
               "réponse exacte) : relancer la cellule ne re-consomme pas d'appels API pour ce "
               "qui a déjà été jugé."); r += 2

    line(r, "Feuille", "Contenu", bold=True); r += 1
    for name, desc in [
        ("Dataset", "Taille du jeu de données scoré et répartition (Axe, Difficulty, intent_group, intent_category)."),
        ("Overview", "Classement des modèles : exactitude, couverture, taux de non-réponse, taux de verdicts non parsables, taux d'erreurs API."),
        ("Acc_<dimension>", "Exactitude de chaque modèle, ventilée par valeur de la dimension (ex: par Axe)."),
        ("N_<dimension>", "Nombre d'items réellement scorés par modèle et par valeur de la dimension (à lire à côté de Acc_<dimension> pour juger la fiabilité d'un pourcentage calculé sur peu d'items)."),
        ("Details", "Chaque item avec la réponse de chaque modèle, le statut du verdict et la justification donnée par le juge (utile pour auditer les cas limites)."),
    ]:
        line(r, name, desc)
        r += 1

    r += 1
    line(r, "Statut possible", "Définition", bold=True); r += 1
    for name, desc in [
        ("manquant", "Le modèle n'a pas donné de réponse (NaN / vide) — pas d'appel au juge."),
        ("correct", "Le juge a rendu le verdict CORRECT."),
        ("incorrect", "Le juge a rendu le verdict INCORRECT."),
        ("non_parse", "La réponse du juge n'a pas pu être interprétée comme CORRECT/INCORRECT (à vérifier dans Details, colonne raison)."),
        ("erreur_api", "L'appel réseau vers le juge a échoué après plusieurs tentatives."),
    ]:
        line(r, name, desc)
        r += 1

    r += 1
    line(r, "Métrique (Overview)", "Définition", bold=True); r += 1
    for name, desc in [
        ("couverture", "n_scored / n_total : part des items pour lesquels le modèle a donné une réponse exploitable (non vide)."),
        ("exactitude", "n_correct / n_scored (verdicts 'correct' uniquement) — calculée UNIQUEMENT sur les items 'scored'."),
        ("taux_manquant", "Part des items sans réponse du tout."),
        ("taux_non_parse_juge", "Part des items où le verdict du juge n'a pas pu être extrait."),
        ("taux_erreur_api", "Part des items où l'appel au juge a échoué techniquement."),
    ]:
        line(r, name, desc)
        r += 1


# ============================== MAIN ==============================

def main():
    df, model_cols = load_eligible_data()
    group_cols_present = [c for c in GROUP_COLS if c in df.columns]
    missing_groups = [c for c in GROUP_COLS if c not in df.columns]
    if missing_groups:
        print(f"(Dimensions absentes de la feuille, ignorées : {missing_groups})")

    print("Jugement des réponses par Gemma 4...")
    results = score_all_models_llm(df, model_cols)

    overview = compute_overview(df, model_cols, results)
    model_order = overview["modele"].tolist()

    dataset_summary = build_dataset_summary(df, group_cols_present)
    details = build_details(df, model_cols, results)

    out_path = Path(OUTPUT_PATH)
    tmp_path = out_path.with_name(out_path.stem + ".tmp" + out_path.suffix)

    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        write_sheet(
            writer, dataset_summary, "Dataset",
            percent_cols=["part"], color_scale_cols=["part"],
        )

        write_sheet(
            writer, overview, "Overview",
            percent_cols=["couverture", "exactitude", "taux_manquant",
                          "taux_non_parse_juge", "taux_erreur_api"],
            color_scale_cols=["exactitude", "couverture"],
            invert_cols=[],
        )

        for group_col in group_cols_present:
            acc_df, n_df = compute_breakdown(df, model_cols, group_col, model_order, results)
            value_cols = [c for c in acc_df.columns if c != "modele"]

            write_sheet(
                writer, acc_df, f"Acc_{group_col}"[:31],
                percent_cols=value_cols, color_scale_cols=value_cols,
            )
            write_sheet(
                writer, n_df, f"N_{group_col}"[:31],
            )

        write_sheet(writer, details, "Details")

        write_notes_sheet(writer, model_cols, group_cols_present)

    tmp_path.replace(out_path)
    print(f"\nTerminé -> {out_path.resolve()}")


try:
    main()
except Exception as e:
    print(f"\nÉCHEC : {e}")
    raise


Pas de filtre par statut appliqué (284 lignes).
-> 284 items éligibles pour le scoring.
-> 11 modèle(s) détecté(s) : ['claude_sonnet5', 'commandr', 'gemini3_6_flash', 'gemma4_26b', 'gemma4_31b', 'gpt5', 'gptoss_20b', 'grok4_20', 'mistral_24b', 'qwen3_5_9b', 'qwen3_6_27b']
Jugement des réponses par Gemma 4...
3124 réponse(s) non vide(s) au total ; 3124 appel(s) réseau nécessaires (le reste vient du cache 'gemma_judge_cache.json').
   ... 25/3124 appels traités (cache sauvegardé)
   ... 50/3124 appels traités (cache sauvegardé)
   ... 75/3124 appels traités (cache sauvegardé)
   ... 100/3124 appels traités (cache sauvegardé)
   ... 125/3124 appels traités (cache sauvegardé)
   ... 150/3124 appels traités (cache sauvegardé)
   ... 175/3124 appels traités (cache sauvegardé)
   ... 200/3124 appels traités (cache sauvegardé)
   ... 225/3124 appels traités (cache sauvegardé)
   ... 250/3124 appels traités (cache sauvegardé)
   ... 275/3124 appels traités (cache sauvegardé)
   ... 300/3124 app